Phase 1: Data Pre-Processing

In [1]:
# Import libraries
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# See what info actually contains
ticker = 'AAPL'
stock = yf.Ticker(ticker)
info = stock.info
print(sorted(info.keys()))

In [ ]:
# Step 1: Fetch raw data to simulate different types of a portfolio
tickers = ['^GSPC','NVDA','QBTS','GOOGL', 'BITF', 'TSLA', 'BBAI', 'RBLX', 'AVGO', 'UNH', 'PM', 'KO', 'PG', 'COST', 'JNJ', 'LLY', 'WMT', 'IONQ', 'RGTI', 'BMNR','OPEN', 'LEU', 'GEV', 'MSTR', 'CCJ', 'SNDK', 'MU']

# Remove '^GSPC' from the tickers list as it's a market index, not an individual stock
tickers = [ticker for ticker in tickers if ticker != '^GSPC']

print(f"Fetching data for: {tickers}...")

# Download last 2 years of data
data = yf.download(tickers, period='4y', group_by='ticker')

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
# Define helper functions for 4-year calculations

def get_historical_metric_series(df_statement, metric_name, num_periods=4):
    """
    Extracts a historical metric series for a given metric from a financial statement DataFrame.
    Assumes df_statement has metrics as index and dates as columns.
    Returns a pandas Series with dates as index and metric values, sorted oldest to newest.
    """
    if df_statement is None or df_statement.empty or metric_name not in df_statement.index:
        return pd.Series([], dtype=float)

    # Get the row for the metric, convert to Series, drop NaNs, and take the last num_periods (oldest first)
    series = df_statement.loc[metric_name].dropna().head(num_periods).sort_index(ascending=True)
    return series

def calculate_cagr(start_value, end_value, num_years):
    """
    Calculates Compound Annual Growth Rate (CAGR) with extended handling for negative values and zero.
    Returns np.nan for uncalculable real growth rates.
    """

    if num_years <= 0:
        return np.nan # Cannot calculate growth for zero or negative periods

    # Handle cases with zero start_value
    if start_value == 0:
        if end_value > 0:
            return float('inf') # Infinite growth from zero to positive
        elif end_value < 0:
            return float('-inf') # Infinite decline from zero to negative
        else: # end_value is also 0
            return 0.0 # No growth

    ratio = end_value / start_value

    if ratio < 0:
        if num_years % 2 == 0: # Even root of a negative number is not real
            return np.nan
        else: # Odd root of a negative number is real and negative
            try:
                # Use np.sign to preserve the sign and abs for the power calculation
                cagr = np.sign(ratio) * (abs(ratio)**(1 / num_years)) - 1
                return cagr
            except Exception:
                return np.nan # Catch any unexpected errors during calculation
    else: # ratio >= 0
        try:
            cagr = (ratio**(1 / num_years)) - 1
            return cagr
        except Exception:
            return np.nan # Catch any unexpected errors during calculation

def calculate_four_year_average_from_series(series):
    """Calculates the average from a given series, ensuring enough data points."""
    if series.empty or len(series) < 2: # Require at least 2 data points for a meaningful average
        return 0.0
    return series.mean()

def calculate_four_year_growth_from_series(series):
    """Calculates the 4-year growth rate (CAGR) from a given series."""
    if series.empty or len(series) < 2:
        return 0.0

    start_value = series.iloc[0]
    end_value = series.iloc[-1]
    num_years = len(series) - 1 # Number of periods for growth calculation (e.g., 4 data points = 3 years of growth)

    return calculate_cagr(start_value, end_value, num_years) if num_years > 0 else 0.0

def get_statement_df(ticker_obj, statement_type, frequency='annual'):
    """Helper to get the correct statement DataFrame based on frequency."""
    if frequency == 'annual':
        if statement_type == 'financials':
            return ticker_obj.financials
        elif statement_type == 'balance_sheet':
            return ticker_obj.balance_sheet
        elif statement_type == 'cashflow':
            return ticker_obj.cashflow
    elif frequency == 'quarterly':
        if statement_type == 'financials':
            return ticker_obj.quarterly_financials
        elif statement_type == 'balance_sheet':
            return ticker_obj.quarterly_balance_sheet
        elif statement_type == 'cashflow':
            return ticker_obj.quarterly_cashflow
    return pd.DataFrame() # Return empty DataFrame if type is unknown or frequency is invalid

def get_annualized_metric_series(ticker_obj, statement_type, aggregation_method='last', num_years=4):
    """
    Aggregates quarterly metric series into annual series.
    'aggregation_method': 'sum' for Income/Cashflow items, 'last' for Balance Sheet items.
    """
    quarterly_series = get_historical_metric_series(
        get_statement_df(ticker_obj, statement_type, frequency='quarterly'),
        metric_name,
        num_periods=num_years * 4 # Get enough quarters for the desired number of annual periods
    )

    if quarterly_series.empty:
        return pd.Series([], dtype=float)

    # Convert quarterly dates to fiscal year-end for grouping
    # Group by year, then aggregate.
    if aggregation_method == 'sum':
        annual_series = quarterly_series.resample('Y').sum().dropna()
    elif aggregation_method == 'last':
        annual_series = quarterly_series.resample('Y').last().dropna()
    else:
        raise ValueError("aggregation_method must be 'sum' or 'last'")

    # Ensure we only have the last `num_years` of annual data, sorted
    return annual_series.tail(num_years)

# --- Specific 4-Year Calculation Functions ---

def calc_four_year_revenue_growth(ticker_obj):
    series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Total Revenue', num_periods=4)
    return calculate_four_year_growth_from_series(series)

def calc_four_year_eps_growth(ticker_obj):
    series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Diluted EPS', num_periods=4)
    return calculate_four_year_growth_from_series(series)

def calc_four_year_fcf_growth(ticker_obj):
    series = get_historical_metric_series(get_statement_df(ticker_obj, 'cashflow'), 'Free Cash Flow', num_periods=4)
    return calculate_four_year_growth_from_series(series)

def calc_four_year_gross_profit_margin_avg(ticker_obj):
    gross_profit_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Gross Profit', num_periods=4)
    revenue_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Total Revenue', num_periods=4)
    aligned_df = pd.DataFrame({'gross_profit': gross_profit_series, 'revenue': revenue_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0
    ratios = aligned_df.apply(lambda row: row['gross_profit'] / row['revenue'] if row['revenue'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

def calc_four_year_operating_profit_margin_avg(ticker_obj):
    op_income_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Operating Income', num_periods=4)
    revenue_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Total Revenue', num_periods=4)
    aligned_df = pd.DataFrame({'op_income': op_income_series, 'revenue': revenue_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0
    ratios = aligned_df.apply(lambda row: row['op_income'] / row['revenue'] if row['revenue'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

def calc_four_year_net_profit_margin_avg(ticker_obj):
    net_income_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Net Income', num_periods=4)
    revenue_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Total Revenue', num_periods=4)
    aligned_df = pd.DataFrame({'net_income': net_income_series, 'revenue': revenue_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0
    ratios = aligned_df.apply(lambda row: row['net_income'] / row['revenue'] if row['revenue'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

def calc_four_year_roe_avg(ticker_obj):
    # For ROE, use annual Net Income and annual Stockholders Equity
    net_income_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Net Income', num_periods=4)
    equity_series = get_historical_metric_series(get_statement_df(ticker_obj, 'balance_sheet'), 'Stockholders Equity', num_periods=4)

    aligned_df = pd.DataFrame({'net_income': net_income_series, 'equity': equity_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0

    ratios = aligned_df.apply(lambda row: row['net_income'] / row['equity'] if row['equity'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

def calc_four_year_roa_avg(ticker_obj):
    net_income_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Net Income', num_periods=4)
    assets_series = get_historical_metric_series(get_statement_df(ticker_obj, 'balance_sheet'), 'Total Assets', num_periods=4)
    aligned_df = pd.DataFrame({'net_income': net_income_series, 'assets': assets_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0
    ratios = aligned_df.apply(lambda row: row['net_income'] / row['assets'] if row['assets'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

def calc_four_year_fcf_margin_avg(ticker_obj):
    fcf_series = get_historical_metric_series(get_statement_df(ticker_obj, 'cashflow'), 'Free Cash Flow', num_periods=4)
    revenue_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'Total Revenue', num_periods=4)
    aligned_df = pd.DataFrame({'fcf': fcf_series, 'revenue': revenue_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0
    ratios = aligned_df.apply(lambda row: row['fcf'] / row['revenue'] if row['revenue'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

def calc_four_year_debt_to_equity_avg(ticker_obj):
    # For Debt to Equity, use annual Total Debt and annual Stockholders Equity
    debt_series = get_historical_metric_series(get_statement_df(ticker_obj, 'balance_sheet'), 'Total Debt', num_periods=4)
    equity_series = get_historical_metric_series(get_statement_df(ticker_obj, 'balance_sheet'), 'Stockholders Equity', num_periods=4)

    aligned_df = pd.DataFrame({'debt': debt_series, 'equity': equity_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0

    ratios = aligned_df.apply(lambda row: row['debt'] / row['equity'] if row['equity'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

def calc_four_year_debt_to_ebitda_avg(ticker_obj):
    debt_series = get_historical_metric_series(get_statement_df(ticker_obj, 'balance_sheet'), 'Total Debt', num_periods=4)
    ebitda_series = get_historical_metric_series(get_statement_df(ticker_obj, 'financials'), 'EBITDA', num_periods=4)
    aligned_df = pd.DataFrame({'debt': debt_series, 'ebitda': ebitda_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0
    ratios = aligned_df.apply(lambda row: row['debt'] / row['ebitda'] if row['ebitda'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

def calc_four_year_current_ratio_avg(ticker_obj):
    # For Current Ratio, use annual Total Current Assets and annual Current Liabilities
    annual_current_assets_series = get_historical_metric_series(get_statement_df(ticker_obj, 'balance_sheet'), 'Current Assets', num_periods=4)
    annual_current_liabilities_series = get_historical_metric_series(get_statement_df(ticker_obj, 'balance_sheet'), 'Current Liabilities', num_periods=4)

    aligned_df = pd.DataFrame({'assets': annual_current_assets_series, 'liabilities': annual_current_liabilities_series}).dropna()
    if aligned_df.empty or len(aligned_df) < 2: return 0.0

    ratios = aligned_df.apply(lambda row: row['assets'] / row['liabilities'] if row['liabilities'] != 0 else np.nan, axis=1).dropna()
    return calculate_four_year_average_from_series(ratios)

In [ ]:
for ticker in tickers:
    stock = yf.Ticker(ticker)
    print(f"\nChecking financial statements for {ticker}:")

    has_financials = False
    try:
        if not stock.financials.empty:
            print(f"  Annual Financials: Available (rows: {len(stock.financials.index)})")
            has_financials = True
        else:
            print("  Annual Financials: Empty")
    except AttributeError:
        print("  Annual Financials: Not available (AttributeError)")
    except Exception as e:
        print(f"  Annual Financials: Error checking ({e})")

    try:
        if not stock.balance_sheet.empty:
            print(f"  Annual Balance Sheet: Available (rows: {len(stock.balance_sheet.index)})")
            has_financials = True
        else:
            print("  Annual Balance Sheet: Empty")
    except AttributeError:
        print("  Annual Balance Sheet: Not available (AttributeError)")
    except Exception as e:
        print(f"  Annual Balance Sheet: Error checking ({e})")

    try:
        if not stock.cashflow.empty:
            print(f"  Annual Cash Flow: Available (rows: {len(stock.cashflow.index)})")
            has_financials = True
        else:
            print("  Annual Cash Flow: Empty")
    except AttributeError:
        print("  Annual Cash Flow: Not available (AttributeError)")
    except Exception as e:
        print(f"  Annual Cash Flow: Error checking ({e})")

    if not has_financials:
        print(f"  Conclusion: {ticker} likely does not have traditional financial statements via yfinance.")

In [ ]:
# Step 2: Strcuture the Data (Handling Multi-Index)
# yfinance returns a complex multi-index dataframe.
# We need to flatten it so each row is a specific Date + Ticker.

portfolio_data = []
print("Fetch Fundamental Data")

# Define a mapping from requested keys to available yfinance info keys
# For keys not directly available, they will default to 0.
key_mapping = {
    'OneYearRevenueGrowth': 'revenueGrowth',
    'FourYearRevenueGrowth': 'CALC_FOUR_YEAR_REVENUE_GROWTH',
    'EPSGrowth': 'CALC_FOUR_YEAR_EPS_GROWTH', # Assume 5-year growth is desired for 'EPSGrowth' when asking for 5-year calculations
    'OneYearFreeCashFlowGrowth': None, # Not directly calculable for 1 year easily
    'FourYearFreeCashFlowGrowth': 'CALC_FOUR_YEAR_FCF_GROWTH',
    'TTMGrossProfits': 'grossProfits',
    'FourYearGrossProfitMargin': 'CALC_FOUR_YEAR_GROSS_PROFIT_MARGIN_AVG',
    'TTMOperatingProfitMargin': 'operatingMargins',
    'FourYearOperatingProfitMargin': 'CALC_FOUR_YEAR_OPERATING_PROFIT_MARGIN_AVG',
    'TTMNetProfitMargin': 'profitMargins',
    'FourYearNetProfitMargin': 'CALC_FOUR_YEAR_NET_PROFIT_MARGIN_AVG',
    'TTMReturnOnEquity': 'returnOnEquity',
    'FourYearReturnOnEquity': 'CALC_FOUR_YEAR_ROE_AVG',
    'TTMReturnOnInvestedCapital': None, # Not feasible from yfinance financials alone
    'FourYearReturnOnInvestedCapital': None, # Not feasible from yfinance financials alone
    'TTMReturnOnAssets': 'returnOnAssets',
    'FourYearReturnOnAssets': 'CALC_FOUR_YEAR_ROA_AVG',
    'TTMFreeCashFlowMargin': None, # Not directly in info
    'FourYearFreeCashFlowMargin': 'CALC_FOUR_YEAR_FCF_MARGIN_AVG',
    'CurrentTotalDebtToEquity': 'debtToEquity',
    'FourYearTotalDebtToEquity': 'CALC_FOUR_YEAR_DEBT_TO_EQUITY_AVG',
    'CurrentTotalDebtToEBITDA': 'enterpriseToEbitda', # Closest available: Enterprise Value to EBITDA from info
    'FourYearTotalDebtToEBITDA': 'CALC_FOUR_YEAR_DEBT_TO_EBITDA_AVG',
    'CurrentRatio': 'currentRatio',
    'FourYearCurrentRatio': 'CALC_FOUR_YEAR_CURRENT_RATIO_AVG',
    'InterestCoverage': 'CALC_FOUR_YEAR_INTEREST_COVERAGE_AVG', # No direct info key, calculate 5-year average
    'CurrentPE': 'trailingPE',
    'FourYearPE': None, # Not feasible from yfinance financials alone (requires historical prices + EPS)
    'CurrentPEG': 'trailingPegRatio',
    'FourYearPEG': None, # Not feasible from yfinance financials alone
}

# Removed the preliminary filtering for valid_tickers_for_fundamentals.
# The error handling within the loop for each ticker will now manage missing data.

for ticker in tickers:
    fundamental_factors = {}
    try:
        stock = yf.Ticker(ticker)
        info = stock.info

        # Creates column from the mapping
        for requested_key, info_key_or_calc_str in key_mapping.items():
            if isinstance(info_key_or_calc_str, str) and info_key_or_calc_str.startswith('CALC_'):
                # Call the appropriate calculation function
                calc_func_name = info_key_or_calc_str.lower().replace('calc_', '') # e.g., 'five_year_revenue_growth'
                calc_func = globals().get('calc_' + calc_func_name) # Get function from global scope
                if calc_func:
                    fundamental_factors[requested_key] = calc_func(stock)
                else:
                    fundamental_factors[requested_key] = 0.0
            elif info_key_or_calc_str: # It's a direct info key
                fundamental_factors[requested_key] = info.get(info_key_or_calc_str, 0.0)
            else: # No direct mapping or calculation possible
                fundamental_factors[requested_key] = 0.0 # Set to 0 if no direct mapping or calc

    except Exception as e:
        print(f"Error fetching fundamental data for {ticker}: {e}")
        # If any error occurs (e.g., ticker not found, no info), set all factors to 0.0
        fundamental_factors = {key: 0.0 for key in key_mapping.keys()}

    # Extract specific ticker data
    df = data[ticker].copy()

    # Add 'Ticker' column to the DataFrame
    df['Ticker'] = ticker

    # Rename 'Close' to 'Close_Price' for clarity
    df.rename(columns={'Close': 'Close_Price'}, inplace=True)

    # Select the relevant columns including 'Ticker'
    df = df[['Close_Price', 'Volume', 'Ticker']]

    # Add fundamental factors to each row of the DataFrame
    for factor, value in fundamental_factors.items():
        df[factor] = value

    portfolio_data.append(df)

# Combine all stocks into one big list
final_df = pd.concat(portfolio_data)

In [ ]:
# List of available metrics on Yahoo finance
sample_ticker = 'NVDA'
stock_nvda = yf.Ticker(sample_ticker)

print(f"Financials Index for {sample_ticker}:")
if not stock_nvda.financials.empty:
    print(stock_nvda.financials.index.tolist())
else:
    print("Financials are empty or not available.")

print(f"\nBalance Sheet Index for {sample_ticker}:")
if not stock_nvda.balance_sheet.empty:
    print(stock_nvda.balance_sheet.index.tolist())
else:
    print("Balance Sheet is empty or not available.")

print(f"\nCash Flow Index for {sample_ticker}:")
if not stock_nvda.cashflow.empty:
    print(stock_nvda.cashflow.index.tolist())
else:
    print("Cash Flow is empty or not available.")

In [ ]:
# Check historical data points
new_tickers = ['IONQ', 'OKLO'] # Examples of potentially newer companies

print("Checking historical data points for newer tickers:")
for ticker_symbol in new_tickers:
    stock = yf.Ticker(ticker_symbol)
    # Check Total Stockholder Equity (used in ROE and Debt-to-Equity)
    equity_series = get_historical_metric_series(get_statement_df(stock, 'balance_sheet'), 'Total Stockholder Equity', num_periods=4)
    print(f"\n{ticker_symbol}: Total Stockholder Equity historical data points (max 5): {len(equity_series)}")
    print(equity_series.index.tolist())

    # Check Total Revenue (used in many calculations)
    revenue_series = get_historical_metric_series(get_statement_df(stock, 'financials'), 'Total Revenue', num_periods=4)
    print(f"{ticker_symbol}: Total Revenue historical data points (max 5): {len(revenue_series)}")
    print(revenue_series.index.tolist())

As you can see from the output, newer companies like IONQ and OKLO often have fewer than 5 (or even 2) annual data points for these financial metrics. When the `get_historical_metric_series` function retrieves fewer than two data points, the subsequent averaging/growth calculation functions return `0.0`, which explains why these fields are populated with zeros.

In [ ]:
sample_ticker_for_detail = 'IONQ'
stock_ionq = yf.Ticker(sample_ticker_for_detail)

# Get Net Income Series
net_income_series_ionq = get_historical_metric_series(get_statement_df(stock_ionq, 'financials'), 'Net Income', num_periods=4)
print(f"\nIONQ Net Income Series:\n{net_income_series_ionq}")

# Get Total Stockholder Equity Series
equity_series_ionq = get_historical_metric_series(get_statement_df(stock_ionq, 'balance_sheet'), 'Total Stockholder Equity', num_periods=4)
print(f"\nIONQ Total Stockholder Equity Series:\n{equity_series_ionq}")

# Align and drop NaNs for ROE calculation
aligned_roe_df_ionq = pd.DataFrame({'net_income': net_income_series_ionq, 'equity': equity_series_ionq}).dropna()
print(f"\nIONQ Aligned DataFrame for ROE (after dropna()):\n{aligned_roe_df_ionq}")
print(f"Number of aligned data points for ROE: {len(aligned_roe_df_ionq)}")

# Calculate ratios
ratios_ionq = aligned_roe_df_ionq.apply(lambda row: row['net_income'] / row['equity'] if row['equity'] != 0 else np.nan, axis=1).dropna()
print(f"\nIONQ ROE Ratios (after handling division by zero and dropna()):\n{ratios_ionq}")
print(f"Number of valid ROE ratios: {len(ratios_ionq)}")

# Final 4-year average ROE
final_roe_ionq = calculate_four_year_average_from_series(ratios_ionq)
print(f"\nIONQ Final 4-Year Average ROE: {final_roe_ionq}")

In [ ]:
# ==========================================
# STEP 3: Feature Engineering (The "Time-Series" Skill)
# ==========================================
# This is crucial for OneRoad. You are creating "Lag" features.
# A model can't see the future, but it can see the past (Moving Averages).

processed_data = []

for ticker_name, group_df in final_df.groupby('Ticker'):
    # Make a copy to avoid SettingWithCopyWarning
    group_df_copy = group_df.copy()

    # Apply feature engineering calculations
    group_df_copy = group_df_copy.assign(
        # 1a. Daily Return (Percentage change from yesterday)
        Daily_Return=group_df_copy['Close_Price'].pct_change(fill_method=None),
        # 1b. Weekly Return (Percentage change from yesterday)
        Weekly_Return=group_df_copy['Close_Price'].pct_change(periods=5, fill_method=None),
        # 1c. Monthly Return (Percentage change from last month)
        Monthly_Return=group_df_copy['Close_Price'].pct_change(periods=21, fill_method=None),
        # Removed Yearly_Return to prevent excessive NaN values for newer stocks
        # Yearly_Return=group_df_copy['Close_Price'].pct_change(periods=252, fill_method=None),
        # 2a. Moving Average (50 days) - identifies the short-term trend
        MA_50=group_df_copy['Close_Price'].rolling(window=50).mean(),
        # 2b. Moving Average (100 days) - identifies the medium-term trend
        MA_100=group_df_copy['Close_Price'].rolling(window=100).mean(),
        # Removed MA_200 to prevent excessive NaN values for newer stocks
        # MA_200=group_df_copy['Close_Price'].rolling(window=200).mean(),
        # 3. Lag Feature (Price 1 day ago) - helps predict "next step"
        Lag_1=group_df_copy['Close_Price'].shift(1)
    )
    processed_data.append(group_df_copy)

# Concatenate all processed dataframes back into final_df
final_df = pd.concat(processed_data).reset_index()

In [ ]:
# ==========================================
# STEP 4: Data Cleaning
# ==========================================
# Rolling averages and other feature engineering calculations create NaN (empty) values.
# Instead of dropping all rows, which can empty the DataFrame for stocks with shorter histories,
# we will fill these NaN values for the engineered features. Fundamental factors are already set to 0.0
# if not available.
print(f"Data shape before cleaning: {final_df.shape}")

# Identify columns that are likely to contain NaNs from feature engineering
# and fill them, for example, with 0.
# This list should include all engineered features plus any others that might consistently have NaNs.
engineered_features = ['Daily_Return', 'Weekly_Return', 'Monthly_Return', 'MA_50', 'MA_100', 'Lag_1']

for col in engineered_features:
    if col in final_df.columns:
        final_df[col] = final_df[col].fillna(0) # Fill NaNs with 0

# Optionally, drop rows where 'Close_Price' (a core data point) is still NaN, if any.
# This is less aggressive than dropping all NaNs.
final_df.dropna(subset=['Close_Price'], inplace=True)

print(f"Data shape after cleaning: {final_df.shape}")

In [ ]:
# Check distribution of 4 year ROA for all stocks
summary_stats = final_df['FourYearReturnOnAssets'].describe
print("--- Using describe() ---")
print(summary_stats)

print("\n--- Using quartile() ---")
Q1 = (final_df['FourYearReturnOnAssets'].quantile([0.25]))
print(f"Q1: {Q1}")

Q2 = (final_df['FourYearReturnOnAssets'].quantile([0.5]))
print(f"Q2: {Q2}")

Q3 = (final_df['FourYearReturnOnAssets'].quantile([0.75]))
print(f"Q3: {Q3}")

print("\n--- Using min() and max() ---")
Minimum = (final_df['FourYearReturnOnAssets'].min())
print(f"Minimum: {Minimum}")

Maximum = (final_df['FourYearReturnOnAssets'].max())
print(f"Maximum: {Maximum}")

mean_value = (final_df['FourYearReturnOnAssets'].mean())
print(f"Mean: {mean_value}")

In [ ]:
# Check historical data points for EPS growth, four year free cash and four year interest coverage

# Check historical data points
new_tickers = ['IONQ', 'BMNR', 'MSTR', 'OKLO', 'OPEN', 'RGTI', 'SMR', 'NVDA', 'SNDK','WDC'] # Examples of potentially newer companies

print("Checking historical data points for newer tickers:")
for ticker_symbol in new_tickers:
    stock = yf.Ticker(ticker_symbol)
    # Check Four Year Revenue Growth
    earnings_series = get_historical_metric_series(get_statement_df(stock, 'financials'), 'Diluted EPS', num_periods=4)
    print(f"\n{ticker_symbol}: Total EPS historical data points (max 4): {len(earnings_series)}")
    print(earnings_series.index.tolist())

In [ ]:
# Obtain the ROE for Apple
ticker = yf.Ticker('AAPL')

# Get the Net Income series for Apple over the last 4 years
aapl_net_income_series = get_historical_metric_series(get_statement_df(ticker, 'financials'), 'Net Income', num_periods=4)
print(f"Apple (AAPL) 4-year Net Income Series:\n{aapl_net_income_series}")

# Get the Shareholder Equity series for Apple over the last 4 years
aapl_equity_series = get_historical_metric_series(get_statement_df(ticker, 'balance_sheet'), 'Stockholders Equity', num_periods=4)
print(f"\nApple (AAPL) 4-year Total Stockholders Equity Series:\n{aapl_equity_series}")

# Align Net Income and Shareholder Equity
aligned_roe_df_aapl = pd.DataFrame({'net_income': aapl_net_income_series, 'equity': aapl_equity_series}).dropna()
print(f"\nApple (AAPL) Aligned DataFrame for ROE (after dropna()):\n{aligned_roe_df_aapl}")
print(f"Number of aligned data points for ROE: {len(aligned_roe_df_aapl)}")

# Calculate ROE for Apple
if not aligned_roe_df_aapl.empty:
    aapl_roe_ratios = aligned_roe_df_aapl.apply(lambda row: row['net_income'] / row['equity'] if row['equity'] !=0 else np.nan, axis=1).dropna()
    print(f"\nApple (AAPL) ROE Ratios (after handling division by zero and dropna()):\n{aapl_roe_ratios}")
    print(f"Number of valid ROE ratios: {len(aapl_roe_ratios)}")

    # Final 4-year calculation for ROE
    final_aapl_roe = calculate_four_year_average_from_series(aapl_roe_ratios)
    print(f"\nFinal Apple (AAPL) ROE Ratios Averaged by 4 years: {final_aapl_roe}")
else:
    print("Not enough aligned data points for Apple ROE calculation.")

In [ ]:
amazon_ticker = yf.Ticker('AMZN')

# Get the Free Cash Flow series for Amazon over the last 4 years
amzn_fcf_series = get_historical_metric_series(get_statement_df(amazon_ticker, 'cashflow'), 'Free Cash Flow', num_periods=4)

print(f"Amazon (AMZN) 4-year Free Cash Flow Series:\n{amzn_fcf_series}")

# Calculate the 4-year growth using the defined function
amzn_fcf_growth = calculate_four_year_growth_from_series(amzn_fcf_series)
print(f"\nCalculated 4-Year Free Cash Flow Growth for Amazon: {amzn_fcf_growth:.4f}")

if not amzn_fcf_series.empty and len(amzn_fcf_series) >= 2:
    start_value = amzn_fcf_series.iloc[0]
    end_value = amzn_fcf_series.iloc[-1]
    num_years = len(amzn_fcf_series) - 1
    print(f"\nTo calculate CAGR (Compound Annual Growth Rate) for Free Cash Flow:")
    print(f"Start value (oldest): {start_value}")
    print(f"End value (most recent): {end_value}")
    print(f"Number of growth periods: {num_years} years")
    if num_years > 0:
        cagr = calculate_cagr(start_value, end_value, num_years)
        print(f"Calculated CAGR: {cagr:.4f}")
    else:
        print("CAGR calculation requires at least two data points.")
else:
    print("Not enough data points to perform a meaningful 4-year growth calculation.")

**Inspecting Moat Scores for Specific Tickers**

Let's inspect the calculated fundamental factors, Moat Scores, and Moat Categories for the mentioned tickers: NVDA, GOOGL, AMZN, and IONQ. This will help us understand why they are receiving their current classifications and pinpoint any discrepancies.

In [ ]:
import numpy as np

def calculate_moat_score(row):
    score = 0

    # Wide Moat Conditions (each met condition adds 2 points)
    # Check for valid numerical values before comparison
    if row['FourYearRevenueGrowth'] > 0.064: score += 2
    if row['EPSGrowth'] > 0.096: score += 2
    if row['FourYearFreeCashFlowGrowth'] > 0.08: score += 2
    if row['FourYearNetProfitMargin'] > 0.12: score += 2
    if row['FourYearGrossProfitMargin'] > 0.40: score += 2 # Using Gross Profit Margin as per latest mapping
    if row['FourYearOperatingProfitMargin'] > 0.144: score += 2
    if row['FourYearReturnOnEquity'] > 0.16: score += 2
    if row['FourYearReturnOnAssets'] > 0.02: score += 2
    if 0 <= row['FourYearTotalDebtToEquity'] < 0.5: score += 2 # Using 'CurrentTotalDebtToEquity'
    if 0 <= row['FourYearTotalDebtToEBITDA'] < 3: score += 2 # Using 'CurrentTotalDebtToEBITDA'
    # Reassessed Current Ratio conditions based on user feedback
    # Wide Moat: Ideal balance of liquidity without excessive idle assets
    if 1.5 <= row['FourYearCurrentRatio'] <= 5.0: score += 2
    # Note: 'FiveYearReturnOnInvestedCapital' and 'FiveYearPE/PEG' were often 0.0 due to yfinance limitations,
    # so they are excluded from scoring for now to avoid penalizing tickers without this data.

    # Narrow Moat Conditions (each met condition adds 1 point if not already covered by a Wide Moat criterion above)
    # To avoid double counting, these are checked against ranges where they might not get 2 points.
    if 0.032 <= row['FourYearRevenueGrowth'] <= 0.064: score += 1
    if 0.048 <= row['EPSGrowth'] <= 0.096: score += 1
    if 0.032 <= row['FourYearFreeCashFlowGrowth'] <= 0.08: score += 1
    if 0.048 <= row['FourYearNetProfitMargin'] <= 0.12: score += 1
    if 0.24 <= row['FourYearGrossProfitMargin'] <= 0.40: score += 1
    if 0.08 <= row['FourYearOperatingProfitMargin'] <= 0.144: score += 1
    if 0.096 <= row['FourYearReturnOnEquity'] <= 0.16: score += 1
    if 0.005 <= row['FourYearReturnOnAssets'] <= 0.02: score += 1
    if 0.5 <= row['FourYearTotalDebtToEquity'] <= 1.5: score += 1
    if 3 <= row['FourYearTotalDebtToEBITDA'] <= 4: score += 1
    # Narrow Moat: Acceptable liquidity, slightly less optimal or a bit too much cash
    if 1.0 <= row['FourYearCurrentRatio'] < 1.5: score += 1

    return score

# Apply the function to each row to get the Moat_Score
final_df['Moat_Score'] = final_df.apply(calculate_moat_score, axis=1)

# Define conditions for np.select based on the accumulated Moat_Score
conditions = [
    final_df['Moat_Score'] >= 21,  # Example score range for Wide Moat
    (final_df['Moat_Score'] >= 13) & (final_df['Moat_Score'] <= 20) # Example score range for Narrow Moat
]

# Define choices for np.select
choices = ['Wide Moat', 'Narrow Moat']

# Apply np.select. Any stock not meeting wide or narrow conditions will default to 'No Moat'
final_df['Moat_Category'] = np.select(conditions, choices, default='No Moat')

# Display a sample to check the results (optional, for verification)
# Display one row for each unique ticker to show a diverse sample
# The display() function should be moved outside the code block as per the environment rules.
display(final_df[final_df['Ticker'] != '^GSPC'].groupby('Ticker')[['Ticker','Moat_Score', 'Moat_Category']].head(1))

In [ ]:
# Define the tickers to inspect
tickers_to_inspect = ['QBTS','BITF','AVGO','COST','SNDK','MU','LEU','OPEN','GOOGL','IONQ','JNJ','KO','LLY','BBAI','RBLX','MSTR','GEV','NVDA','CCJ','PG','PM','BMNR','RGTI','TSLA','UNH','WMT']

# Get all columns that were used in the calculate_moat_score function plus the final scores
moat_related_columns = [
    'Ticker',
    'FourYearRevenueGrowth', 'EPSGrowth', 'FourYearFreeCashFlowGrowth',
    'FourYearNetProfitMargin', 'FourYearGrossProfitMargin', 'FourYearOperatingProfitMargin',
    'FourYearReturnOnEquity', 'FourYearReturnOnAssets', 'FourYearTotalDebtToEquity',
    'FourYearTotalDebtToEBITDA', 'FourYearCurrentRatio',
    'Moat_Score', 'Moat_Category'
]

# Filter final_df for the tickers and relevant columns, and display one entry per ticker
# We take the first entry for each ticker, as moat-related scores are generally constant per ticker after calculation
# And we also only want to show the specific columns that are moat-related.
inspection_df = final_df[final_df['Ticker'].isin(tickers_to_inspect)].groupby('Ticker')[moat_related_columns].head(1)

display(inspection_df)

In [ ]:
# Save the inspection_df to a CSV file
output_csv_path = 'moat_scores_inspection.csv'
inspection_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')

print(f"Inspection DataFrame saved to {output_csv_path}")

In [ ]:
# ==========================================
# STEP 5: Normalization (Scaling)
# ==========================================
# ML models struggle when one number is 150 (AAPL) and another is 3000 (GOOGL).
# We scale everything between 0 and 1.

scaler = MinMaxScaler()

# We only scale the numerical features the model will learn from
# Updated to reflect currently generated features
features_to_scale = [
    'Close_Price', 'Volume', 'MA_50', 'MA_100', 'Daily_Return', 'Weekly_Return', 'Monthly_Return', 'Lag_1',
    'OneYearRevenueGrowth', 'FourYearRevenueGrowth', 'EPSGrowth', 'FourYearFreeCashFlowGrowth',
    'FourYearFreeCashFlowGrowth', 'TTMGrossProfits', 'FourYearGrossProfitMargin', 'TTMOperatingProfitMargin',
    'FourYearOperatingProfitMargin', 'TTMNetProfitMargin', 'FourYearNetProfitMargin', 'TTMReturnOnEquity',
    'FourYearReturnOnEquity', 'TTMReturnOnInvestedCapital', 'FourYearReturnOnInvestedCapital',
    'TTMReturnOnAssets', 'FourYearReturnOnAssets', 'TTMFreeCashFlowMargin', 'FourYearFreeCashFlowMargin',
    'CurrentTotalDebtToEquity', 'FourYearTotalDebtToEquity', 'CurrentTotalDebtToEBITDA',
    'FourYearTotalDebtToEBITDA', 'CurrentRatio', 'FourYearCurrentRatio', 'InterestCoverage',
    'CurrentPE', 'FourYearPE', 'CurrentPEG', 'FourYearPEG',
    'Moat_Score' # Add Moat_Score to features to scale
]

# Filter out features that might not exist after data cleaning or for specific tickers
actual_features_to_scale = [f for f in features_to_scale if f in final_df.columns]

final_df[actual_features_to_scale] = scaler.fit_transform(final_df[actual_features_to_scale])

### Imports Libraries
This cell imports the necessary Python libraries:
- `yfinance` as `yf`: Used to fetch historical stock market data.
- `pandas` as `pd`: A powerful library for data manipulation and analysis, especially with DataFrames.
- `numpy` as `np`: Provides support for large, multi-dimensional arrays and matrices, along with mathematical functions.
- `sklearn.preprocessing.MinMaxScaler`: Used for scaling features to a specified range (typically 0 to 1), which is crucial for many machine learning algorithms.

### Step 1: Fetch Raw Data
This cell defines a list of stock tickers (`NVDA`, `AAPL`, `GOOG`, `AMZN`) and then uses `yfinance.download()` to fetch their historical data. It retrieves the last two years of data for these tickers, with `group_by='ticker'` ensuring the data is organized by each stock.

### Step 6: Model Selection, Training, and Evaluation

**Using the elbow method to find the optimal number of clusters **

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler # For Ridge regression

# Ensure the DataFrame is sorted by Ticker and then by date index for correct time-series operations
# 'index' column from the reset_index is the original date, which is fine.
# It's important to use the original date column if it exists, not a new integer index.
final_df = final_df.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)

# Identify features to exclude from X:
# - 'Ticker': Identifier, not a numerical feature for the model.
# - 'Date': Original date index, not a direct feature for the model.
# - 'Close_Price': The current day's close price; we are predicting the *next* day's close.
# - 'Target_Close_Price': This is our target variable.
# - 'Moat_Category': This will be one-hot encoded, so we remove the original string column.
# - Any features that ended up being '0.0' for all rows due to data scarcity or calculation issues, if they are not intended.
#   For robustness, let's also remove columns that became all-zero after scaling (if any).

# Get a list of all columns in final_df
all_cols = final_df.columns.tolist()

# Columns that should definitely be excluded for now
exclude_for_model_training = [
    'Ticker',
    'Date',
    'Close_Price',
    'Moat_Category',
]

# Columns that should be included for training
# 1. Predictability - FourYearOperatingProfitMargin, FourYearNetProfitMargin, FourYearGrossProfitMargin and FourYearFreeCashFlowGrowth
# 2. Profitability - FourYearOperatingProfitMargin, FourYearNetProfitMargin, FourYearReturnOnEquity, FourYearReturnOnAssets
# 3. Growth - FourYearRevenueGrowth, EPSGrowth, FourYearFreeCashFlowGrowth
# 4. Financial Strength - FourYearTotalDebtToEquity, FourYearTotalDebtToEBITDA, FourYearInterestCoverage, FourYearCurrentRatio
include_for_model_training = [
    'FourYearOperatingProfitMargin',
    'FourYearNetProfitMargin',
    'FourYearGrossProfitMargin',
    'FourYearFreeCashFlowGrowth',
    'FourYearReturnOnEquity',
    'FourYearReturnOnAssets',
    'FourYearRevenueGrowth',
    'EPSGrowth',
    'FourYearTotalDebtToEquity',
    'FourYearTotalDebtToEBITDA',
    'FourYearCurrentRatio'
]

# Separate features for X and y
X = final_df.drop(columns=exclude_for_model_training)
X = final_df[include_for_model_training]

# Filter out columns that do not exist in the DataFrame
exclude_for_model_training = [col for col in exclude_for_model_training if col in all_cols]

# Drop columns from X that contain only NaNs or a single unique value (e.g., all zeros after scaling/imputation)
# This helps prevent errors in models and removes uninformative features.
X = X.loc[:, (X != X.iloc[0]).any()]

# Ensure columns are flattened if they are MultiIndex (can happen if yf.download output wasn't fully flattened)
if isinstance(X.columns, pd.MultiIndex):
    X.columns = X.columns.map(lambda x: '_'.join(str(i) for i in x) if isinstance(x, tuple) else str(x))

# Ensure all remaining columns in X are numeric
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce')
X.fillna(0, inplace=True) # Fill any NaNs introduced by to_numeric with 0

# Dictionary to store model performance
model_performance = {}

# Normalize (scaling) X_train for K-Means Clustering
scaler_kmeans = StandardScaler()
scaled_X = scaler_kmeans.fit_transform(X) # Use scaled_X_train for consistency

# 1. K-Means Clustering
# 1A. Using the elbow method to find the optimal number of clusters
print("\nTraining K-Means Clustering...")

wcss = []
for i in range(1, 25):
  kmeans = KMeans(n_clusters = i, init = 'k-means++', random_state = 42)
  kmeans.fit(scaled_X) # Fit K-Means on scaled X_train
  wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1,25), wcss, marker='o')
plt.title('The Elbow Method')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

# 1B. Training the K-Means Model on the dataset
k = 5
kmeans = KMeans(n_clusters = k, init = 'k-means++', random_state = 42, n_init = 10)
clusters = kmeans.fit_predict(scaled_X) # Predict clusters on scaled X_train

# Create a DataFrame for interpretation, adding clusters and Ticker info
X_train_interpret = X.copy()
X_train_interpret['Clusters'] = clusters
X_train_interpret['Ticker'] = final_df.loc[X.index, 'Ticker'] # Get Ticker from original final_df

# 2. Interpret the Results
summary = X_train_interpret.groupby('Clusters')[include_for_model_training].mean()
print("\n----Cluster Profiles (The AI's Analysis)----")
print(summary)
print("\n")

# 2a. Print which companies fell into which bucket
for i in range(k):
  print(f"Cluster {i}:")
  print(X_train_interpret[X_train_interpret['Clusters'] == i]['Ticker'].unique())
  print("\n")

# 1C. Visualising the clusters
# PCA for 2D visualization
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_X) # PCA on scaled X_train

plt.figure(figsize=(10, 6))
scatter = plt.scatter(pca_result[:, 0], pca_result[:, 1],
                     c=clusters, cmap='viridis', s=50, alpha=0.7)

# Transform centroids to PCA space
pca_centroids = pca.transform(kmeans.cluster_centers_)

# Plot centroids individually with distinct colors and labels
colors = plt.cm.viridis(np.linspace(0, 1, k)) # Generate k distinct colors from the viridis colormap
for i in range(k):
    plt.scatter(pca_centroids[i, 0], pca_centroids[i, 1],
                marker='X', s=200, c=[colors[i]], edgecolor='black',
                label=f'Centroid Cluster {i}')

plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title(f'Time Series Clusters (k={k})')
plt.legend() # Add legend for centroids
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# ==========================================
# Create a Time-Series Analysis for stocks within 4 years
# Download last 2 years of data
data = yf.download(tickers, period='4y', group_by='tickers').xs('Close', level='Price', axis=1)

# Handle NaNs: Drop columns (tickers) with any NaN values
# This ensures that all time series passed to KMeans are complete.
data.dropna(axis=1, inplace=True)

# Include for model training


# If no data remains after dropping NaNs, handle this case
if data.empty:
    print("No complete time series data available for clustering after dropping NaNs.")
    # Exit or handle gracefully
else:
    # ==========================================
    # Normalize the shapes using StandardScaler
    # Transpose: Rows = Stocks, Columns = Days
    X_to_scale = data.T.values # Rows are tickers, columns are dates

    # Scale: Mean=0, Variance=1 for EACH stock.
    # This makes a flat line look different from a volatile line, regardless of price.
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_to_scale) # Correctly scale the time series data

    # ==========================================
    # Time-Series Clustering (K-Means)
    # We cluster the "Rows" (Stocks) based on their "Columns" (Daily Price Movements)
    k = 4
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)


    # ==========================================
    # Visualize the Patterns
    # Plot the "Average Shape" of each cluster to see what the AI found.
    plt.figure(figsize=(15, 10))

    for cluster_num in range(k):
        plt.subplot(2, 2, cluster_num + 1)

        cluster_indices = [i for i, x in enumerate(labels) if x == cluster_num]
        stocks_in_cluster = [data.columns[i] for i in cluster_indices]

        # Generate a colormap for stocks within this specific cluster for distinct colors
        if len(stocks_in_cluster) > 0:
            colors_for_cluster = plt.cm.get_cmap('tab20', len(stocks_in_cluster))
            ticker_to_color_in_cluster = {ticker: colors_for_cluster(j) for j, ticker in enumerate(stocks_in_cluster)}

            # 1. Plot all stocks in this cluster with distinct colors and labels
            for i in cluster_indices:
                ticker_name = data.columns[i]
                plt.plot(X_scaled[i], color=ticker_to_color_in_cluster[ticker_name], alpha=0.7, label=ticker_name)

        # 2. Plot the CLUSTER CENTER (The Representative Pattern)
        plt.plot(kmeans.cluster_centers_[cluster_num], color='red', linewidth=3, label='Cluster Pattern')
        plt.title(f"Cluster {cluster_num}: {len(cluster_indices)} Stocks")
        plt.grid(True, alpha=0.3)
        plt.xticks([]) # Hide dates for cleanliness
        plt.legend(loc='best', fontsize='small') # Add legend specific to this subplot

    plt.suptitle("Discovered Market Regimes (Time-Series Shapes)", fontsize=16)
    plt.tight_layout()
    plt.show()


    # ==========================================
    # Create a dataframe to show which stock belongs to which pattern
    results = pd.DataFrame({'Ticker': data.columns, 'Pattern_Cluster': labels})
    print(results.sort_values('Pattern_Cluster'))

### Step 2: Structure the Data (Handling Multi-Index)
`yfinance` often returns data with a complex multi-index. This cell processes the raw data:
1. It iterates through each ticker.
2. For each ticker, it extracts its specific data and adds a 'Ticker' column to identify the stock.
3. It selects only the 'Close' price and 'Volume' columns.
4. It renames these columns to 'Close_Price' and 'Volume' for clarity.
5. Finally, all individual stock DataFrames are combined into a single `final_df`.

### Step 3: Feature Engineering
This is a critical step for time-series analysis, creating new features from existing data to help a model learn patterns:
- **`Daily_Return`**: Calculates the percentage change in 'Close_Price' from the previous day. This indicates daily gains or losses.
- **`MA_30` (Moving Average)**: Computes the 30-day rolling mean of the 'Close_Price'. This helps to smooth out price fluctuations and identify trends.
- **`Lag_1`**: Creates a feature representing the 'Close_Price' from the previous day (`shift(1)`). This is a common time-series feature that can help predict the next day's price based on the immediate past.

### Step 4: Data Cleaning
Feature engineering steps like calculating moving averages (`MA_30`) or daily returns (`Daily_Return`) often introduce `NaN` (Not a Number) values at the beginning of the series because there isn't enough preceding data to compute them. This cell drops all rows containing any `NaN` values, ensuring that the dataset is complete for subsequent modeling.

### Step 5: Normalization (Scaling)
Machine learning models perform better when input features are on a similar scale. This cell uses `MinMaxScaler` from `sklearn` to scale specific numerical features (`Close_Price`, `Volume`, `MA_30`, `Lag_1`) so that their values fall between 0 and 1. This prevents features with larger numerical ranges from dominating the learning process.